# 04 — Severity Model: How Bad Is a Crash, Given That One Happens?

## Why this notebook exists

The routing engine needs the expected harm of traversing a segment:
segment_risk(s, t) = P(crash | traversal) × E[harm | crash]
↑ ↑
frequency model THIS NOTEBOOK
(needs exposure data) (no exposure needed)
These two factors **multiply**. Summing them would erase the distinction between
an arterial with few but fatal crashes and a junction with many but minor ones.

This notebook builds the second factor, and it is the only one that can be built
honestly from the Unfallatlas alone.

## Why severity is answerable and frequency is not

Accident counts are not risk. Mitte records the most bicycle crashes in Berlin
because Mitte has the most cyclists. Any claim of the form "X is riskier than Y"
needs a denominator of cycling exposure, which this dataset does not contain.

There is one exception: **comparisons conditional on a crash having occurred.**
When both numerator and denominator are crashes — "given a crash, how often is
the outcome severe?" — no exposure data is required.

Every finding in this notebook is of that form, which is what makes it defensible
while the frequency side waits for counter data.

## What this notebook fixes

Our earlier occurrence model learned almost nothing from the EDA. After removing
leaking features it retained four usable inputs — road class, edge length, speed
limit, cycleway presence — and reproduced a fact nobody needed a model for:
crashes concentrate on main roads.

None of the actual findings reached it. Not junctions (80.8% of crashes), not
night (18.79% KSI against 11.70% in the morning), not solo falls (22.01% KSI),
not truck turning conflicts (1.1% of crashes, 31% of deaths).

The reason is that all of those are severity findings, and they cannot transfer
into an occurrence model. This notebook is where they belong.

---
## 1. Data

Two inputs, both already produced upstream:

| File | Contents |
|---|---|
| `berlin_accidents_snapped_to_edges.csv` | 37,896 crashes matched to OSM edges within 25 m |
| `berlin_osm_edge_features.csv` | per-edge road class, speed limit, cycleway flag, length |

**Outcome: KSI** (killed or seriously injured, `UKATEGORIE ∈ {1, 2}`). This is the
standard road-safety outcome and the metric Berlin's Vision Zero targets are
stated in. Fatalities alone are too rare for subgroup analysis — 74 cases over
eight years across 12 districts averages 0.77 per district-year. KSI gives 4,930
cases.

Baseline: **12.99%** of Berlin bicycle crashes result in KSI.

In [5]:
import numpy as np
import pandas as pd

acc = pd.read_csv("../data/processed/berlin_accidents_snapped_to_edges.csv")
edges = pd.read_csv("../data/processed/berlin_osm_edge_features.csv")

acc = acc.merge(
    edges[["edge_uid", "highway_simple", "maxspeed_num", "has_cycleway", "edge_length_m"]],
    on="edge_uid", how="left",
)

# KSI: killed or seriously injured. The standard road-safety outcome and the
# metric Berlin's Vision Zero targets are stated in. Fatalities alone (74 over
# eight years) are too rare for subgroup analysis.
acc["is_ksi"] = acc["accident_severity"].isin([1, 2]).astype(int)

print(f"{len(acc):,} crashes | KSI rate {acc['is_ksi'].mean():.4f}")
print(f"unmatched edges: {acc['highway_simple'].isna().sum():,}")
print(acc["year"].value_counts().sort_index().to_string())

37,896 crashes | KSI rate 0.1299
unmatched edges: 0
year
2018    5182
2019    5000
2020    5105
2021    4287
2022    4653
2023    4465
2024    4446
2025    4758


---
## 2. Junction proximity

80.8% of Berlin's bicycle crashes occur within 20 m of an intersection, and 60%
within 10 m. The edge-based risk model attributes those crashes to whichever
approach arm they happened to snap to, which loses the fact that the conflict
belongs to the junction rather than the link.

Two features are derived here:

- **`near_junction`** — within 20 m of a graph node
- **`node_degree`** — number of arms at that junction. A four-way crossing is a
  different conflict geometry from a bend, and the truck finding lives here:
  52.2% of truck crashes are turning conflicts.

The graph is loaded once and the result cached, because `nearest_nodes` over
37,896 points against 441,544 edges is the slowest step in the pipeline.

In [6]:
import os
import osmnx as ox
import geopandas as gpd

CACHE = "../data/processed/accident_node_dist.csv"

if os.path.exists(CACHE):
    nd = pd.read_csv(CACHE)
else:
    Gp = ox.load_graphml("../data/processed/berlin_bike_network_projected.graphml")
    geom = gpd.GeoSeries(
        gpd.points_from_xy(acc["longitude"], acc["latitude"]), crs="EPSG:4326"
    ).to_crs(Gp.graph["crs"])

    nodes, dist = ox.nearest_nodes(
        Gp, X=geom.x.to_numpy(), Y=geom.y.to_numpy(), return_dist=True
    )
    # Junction arm count: a 4-way crossing behaves differently from a bend.
    degree = dict(Gp.degree())
    nd = pd.DataFrame({
        "edge_uid": acc["edge_uid"],
        "node_dist_m": np.asarray(dist),
        "node_degree": [degree.get(n, 0) for n in nodes],
    })
    nd.to_csv(CACHE, index=False)

acc["node_dist_m"] = nd["node_dist_m"].to_numpy()
acc["node_degree"] = nd["node_degree"].to_numpy()
acc["near_junction"] = (acc["node_dist_m"] <= 20).astype(int)

print(f"near a junction: {acc['near_junction'].mean():.1%}")
print("\nKSI rate by junction proximity:")
print(acc.groupby("near_junction")["is_ksi"].agg(["size", "mean"]).round(4).to_string())
print("\nKSI rate by junction arm count:")
print(acc[acc["near_junction"] == 1].groupby("node_degree")["is_ksi"]
      .agg(["size", "mean"]).round(4).to_string())

near a junction: 80.8%

KSI rate by junction proximity:
                size    mean
near_junction               
0               7261  0.1385
1              30635  0.1279

KSI rate by junction arm count:
              size    mean
node_degree               
1              118  0.1186
2              289  0.1592
3             1263  0.1306
4             8436  0.1285
5             1578  0.1324
6            14152  0.1263
7              798  0.1228
8             3754  0.1281
9               86  0.1512
10             135  0.1259
11              16  0.0000
12              10  0.2000


### 2.1 Junction proximity does not predict severity

| | n | KSI rate |
|---|---|---|
| Away from a junction | 7,261 | 13.85% |
| Within 20 m of a junction | 30,635 | 12.79% |

Junction proximity is associated with *slightly lower* severity, not higher — a
1.1 point difference in the opposite direction to what we expected. Arm count
carries no signal either: KSI rate sits between 12.3% and 13.3% across every
node degree with a usable sample size.

The plausible reading is that speeds are lower at junctions, while mid-link
crashes involve overtaking and running off the carriageway at higher impact
energy.

**This separates two findings that we had conflated.** That 80.8% of crashes occur
near junctions is a statement about *where* crashes happen — it belongs to the
frequency model. It says nothing about how severe they are.

It also strengthens the truck finding by elimination: truck turning crashes are
not lethal because they happen at junctions, since junctions are severity-neutral.
The determining factor is the vehicle.

---
## 3. Feature selection: what is knowable before the trip

The single most important design decision in this notebook.

A route planner knows **when** and **where** the rider will be. It does not know
how a crash would unfold or which vehicle would be involved. So the columns
recorded *after* a crash are excluded, even though they are highly predictive.

| Available at prediction time | Recorded only after the crash |
|---|---|
| `hour`, `is_night`, `is_rush_hour`, `is_weekend`, `season` | `accident_kind` (UART) |
| `maxspeed_num`, `has_cycleway`, `edge_length_m` | `accident_subtype` (UTYP1) |
| `highway_simple` | `is_truck`, `is_car`, `is_pedestrian` |
| `near_junction`, `node_degree` | |

Including `accident_subtype` would raise every metric substantially and make the
model undeployable: the user asking for a route does not know that their trip will
end in a turning conflict with a truck.

This is the mistake that produced our earlier ROC-AUC of 0.973.

**Split: 2018–2023 train, 2024–2025 test.** Random k-fold would leak — crashes at
the same junction would land in both folds — and a forward split also matches how
the model would actually be used.

In [7]:
# Everything here is knowable before the trip starts. A route planner knows when
# and where the rider will be; it does not know which vehicle would hit them.
DEPLOY_NUM = ["hour", "is_night", "is_rush_hour", "is_weekend",
              "maxspeed_num", "has_cycleway", "edge_length_m",
              "near_junction", "node_degree"]
DEPLOY_CAT = ["highway_simple", "season"]

TARGET = "is_ksi"

# Forward split. Random k-fold would leak: crashes at the same junction would
# land in both folds.
train = acc[acc["year"] <= 2023]
test = acc[acc["year"] >= 2024]

print(f"train 2018-2023: {len(train):,}  KSI {train[TARGET].mean():.4f}")
print(f"test  2024-2025: {len(test):,}  KSI {test[TARGET].mean():.4f}")

train 2018-2023: 28,692  KSI 0.1336
test  2024-2025: 9,204  KSI 0.1185


---
## 4. Models and calibration

Four candidates against a prior-only baseline. The comparison is deliberately
modest in scope: with 11 features and 30,000 training rows, tree ensembles land
close together, and the interesting question is not which learner wins.

**Metrics chosen for a rare, probabilistic outcome:**

- **PR-AUC and lift over base rate.** Accuracy is omitted: predicting "no KSI" for
  everyone scores 87% and is useless. Lift of 1.00 means no better than chance.
- **Brier score**, because the routing engine consumes probabilities directly as a
  cost, not thresholded labels. A model that ranks well but is poorly calibrated
  produces a route cost that cannot be interpreted.

**No class weighting.** `class_weight="balanced"` improves apparent recall but
inflates predicted probabilities, which is exactly wrong when the output feeds a
cost function. Isotonic calibration is applied to the best ranker instead, and the
calibration curve is reported so the distortion is visible rather than assumed
away.

In [8]:
import math

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build(estimator):
    pre = ColumnTransformer([
        ("num", StandardScaler(), DEPLOY_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), DEPLOY_CAT),
    ])
    return Pipeline([("pre", pre), ("clf", estimator)])


def evaluate(name, y_true, y_prob):
    """Metrics for a rare, probabilistic outcome. Accuracy is omitted: predicting
    'no KSI' for everyone scores 87% and is useless. Brier matters because the
    routing engine consumes probabilities, not labels."""
    base = y_true.mean()
    ap = average_precision_score(y_true, y_prob)
    return {
        "model": name,
        "base": round(base, 4),
        "pr_auc": round(ap, 4),
        "lift": round(ap / base, 2),
        "roc_auc": round(roc_auc_score(y_true, y_prob), 4),
        "brier": round(brier_score_loss(y_true, y_prob), 4),
    }


cols = DEPLOY_NUM + DEPLOY_CAT
Xtr, ytr = train[cols], train[TARGET]
Xte, yte = test[cols], test[TARGET]

CANDIDATES = {
    "baseline (prior)": DummyClassifier(strategy="prior"),
    "logistic": LogisticRegression(max_iter=2000),
    "random_forest": RandomForestClassifier(
        n_estimators=400, max_depth=12, min_samples_leaf=20,
        random_state=42, n_jobs=-1),
    "hist_gb": HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.08, random_state=42),
}

results, fitted = [], {}
for name, est in CANDIDATES.items():
    if name.startswith("baseline"):
        m = est.fit(Xtr[DEPLOY_NUM], ytr)
        prob = m.predict_proba(Xte[DEPLOY_NUM])[:, 1]
    else:
        m = build(est).fit(Xtr, ytr)
        prob = m.predict_proba(Xte)[:, 1]
        fitted[name] = m
    results.append(evaluate(name, yte.to_numpy(), prob))
    print(f"done: {name}")

# Isotonic calibration on the best ranker. Class weighting is deliberately not
# used: it inflates predicted probabilities, and the routing cost consumes them
# directly rather than thresholding them.
best = max((r for r in results if not r["model"].startswith("baseline")),
           key=lambda r: r["pr_auc"])["model"]
cal = CalibratedClassifierCV(fitted[best], method="isotonic", cv=3).fit(Xtr, ytr)
prob_cal = cal.predict_proba(Xte)[:, 1]
results.append(evaluate(f"{best} + isotonic", yte.to_numpy(), prob_cal))

print()
print(pd.DataFrame(results).to_string(index=False))

frac_pos, mean_pred = calibration_curve(yte, prob_cal, n_bins=10, strategy="quantile")
print("\nCalibration, best model after isotonic:")
for p, o in zip(mean_pred, frac_pos):
    print(f"  predicted {p:.3f} -> observed {o:.3f}")

done: baseline (prior)
done: logistic
done: random_forest
done: hist_gb

              model   base  pr_auc  lift  roc_auc  brier
   baseline (prior) 0.1185  0.1185  1.00   0.5000 0.1047
           logistic 0.1185  0.1393  1.18   0.5470 0.1046
      random_forest 0.1185  0.1342  1.13   0.5391 0.1047
            hist_gb 0.1185  0.1366  1.15   0.5350 0.1047
logistic + isotonic 0.1185  0.1373  1.16   0.5442 0.1048

Calibration, best model after isotonic:
  predicted 0.098 -> observed 0.090
  predicted 0.112 -> observed 0.126
  predicted 0.121 -> observed 0.103
  predicted 0.126 -> observed 0.107
  predicted 0.128 -> observed 0.117
  predicted 0.134 -> observed 0.113
  predicted 0.142 -> observed 0.122
  predicted 0.158 -> observed 0.147
  predicted 0.207 -> observed 0.158


In [9]:
# The night effect is real but too small a slice to lift aggregate performance.
best_model = fitted["logistic"]
test_out = test.copy()
test_out["prob"] = best_model.predict_proba(test[cols])[:, 1]

print("Predicted vs observed KSI by hour band:")
bands = {"night 22-4": (test_out["hour"] >= 22) | (test_out["hour"] <= 4),
         "morning 7-9": test_out["hour"].between(7, 9),
         "midday 11-14": test_out["hour"].between(11, 14),
         "evening 16-18": test_out["hour"].between(16, 18)}
for lbl, m in bands.items():
    s = test_out[m]
    print(f"  {lbl:15s} n={len(s):5,}  predicted {s['prob'].mean():.3f}  "
          f"observed {s['is_ksi'].mean():.3f}")

coef = pd.Series(
    best_model.named_steps["clf"].coef_[0],
    index=best_model.named_steps["pre"].get_feature_names_out(),
).sort_values(key=abs, ascending=False)
print("\nLargest coefficients:")
print(coef.head(12).round(3).to_string())

Predicted vs observed KSI by hour band:
  night 22-4      n=  424  predicted 0.184  observed 0.153
  morning 7-9     n=1,673  predicted 0.125  observed 0.116
  midday 11-14    n=2,294  predicted 0.137  observed 0.119
  evening 16-18   n=2,294  predicted 0.122  observed 0.121

Largest coefficients:
cat__highway_simple_path             0.587
cat__highway_simple_living_street   -0.574
cat__highway_simple_residential     -0.358
cat__highway_simple_secondary       -0.265
cat__highway_simple_other           -0.251
cat__highway_simple_service         -0.235
cat__highway_simple_tertiary        -0.201
cat__highway_simple_primary         -0.158
cat__season_summer                   0.137
cat__season_spring                   0.117
num__is_night                        0.102
num__maxspeed_num                    0.078


In [10]:
print("KSI rate by highway class, with counts:")
print(acc.groupby("highway_simple")["is_ksi"]
      .agg(n="size", ksi="sum", rate="mean")
      .assign(rate=lambda d: (d["rate"] * 100).round(2))
      .sort_values("rate", ascending=False).to_string())

KSI rate by highway class, with counts:
                    n   ksi   rate
highway_simple                    
path               76    22  28.95
trunk              14     4  28.57
primary          5641   827  14.66
tertiary         6300   885  14.05
other             420    58  13.81
secondary       10852  1479  13.63
service           693    81  11.69
residential     12129  1373  11.32
cycleway         1455   163  11.20
living_street     316    31   9.81


In [11]:
# Is the decline across the split gradual or a break? A trend suggests genuine
# improvement; a step change suggests a recording artefact.
print(acc.groupby("year")["is_ksi"]
      .agg(n="size", ksi="sum", rate="mean")
      .assign(rate=lambda d: (d["rate"] * 100).round(2))
      .to_string())

         n  ksi   rate
year                  
2018  5182  743  14.34
2019  5000  644  12.88
2020  5105  677  13.26
2021  4287  582  13.58
2022  4653  619  13.30
2023  4465  567  12.70
2024  4446  533  11.99
2025  4758  558  11.73


---
## 5. Findings

### 5.1 Severity is largely unpredictable from route-planning features

| Model | Lift over base rate | ROC-AUC | Brier |
|---|---|---|---|
| Baseline (prior) | 1.00 | 0.500 | 0.1047 |
| Logistic | **1.18** | 0.547 | 0.1046 |
| HistGradientBoosting | 1.15 | 0.535 | 0.1047 |
| Random forest | 1.13 | 0.539 | 0.1047 |

No model separates outcomes meaningfully. ROC-AUC between 0.535 and 0.547 against
0.500 for chance, and Brier scores identical to the baseline to four decimal
places — predicting a constant 11.85% for every crash performs as well as any
model. That logistic regression edges out both tree ensembles suggests there is no
non-linear structure to find, only noise for the trees to fit.

**This is a negative result, and it is informative.** Given that a crash occurs,
how badly it ends depends largely on circumstances not observable in advance:
vehicle mass, impact angle, whether the rider's head strikes the kerb. Road class,
speed limit, junction proximity and time of day account for almost none of it.

**Consequence:** the severity factor can be treated as approximately constant.
Replacing the fixed `SEVERITY_KSI` weights with a learned model gains nothing, so
the risk formula collapses toward the frequency term:
risk(s) ≈ frequency(s) × constant
### 5.2 Severity follows the speed environment

Excluding classes with fewer than 300 crashes:

| Class | Crashes | KSI rate |
|---|---|---|
| `primary` | 5,641 | **14.66%** |
| `tertiary` | 6,300 | 14.05% |
| `secondary` | 10,852 | 13.63% |
| `residential` | 12,129 | 11.32% |
| `cycleway` | 1,455 | 11.20% |
| `living_street` | 316 | **9.81%** |

Arterials carry 1.29× the KSI rate of residential streets, and the ordering is
monotone in speed environment: `living_street` (7 km/h) lowest, `primary` highest.
This supports H1 at segment level, where district-level data could not — 37,896
crashes rather than 400, against a chi-square that had failed to reject the null
(p = 0.175).

`cycleway` at 11.20% is worth noting separately: conditional on a crash occurring,
outcomes on cycleways are as mild as on residential streets and 24% below
arterials. How *often* crashes occur there is a different question requiring
exposure data.

### 5.3 The `path` coefficient is an artefact and is withdrawn

The logistic model assigned `path` the largest coefficient (+0.587) — but on 76
crashes. `trunk` shows 28.57% KSI on 14. With ten classes ranked, selecting the
highest is a multiple-comparison problem, and neither supports a claim.

This matters because the frequency model scores `path` at 0.016, the lowest of any
class, so the router currently treats park paths as the safest option available.
That remains a concern — but on exposure grounds, not observed severity: near-zero
recorded crashes on an unlit trail at midnight reflects an absence of riders, not
an absence of danger.

### 5.4 Aggregate severity findings hold but do not aggregate into predictive power

Night crashes carry 18.79% KSI against 11.70% in the morning, and truck
involvement 27.25% against 10.59% for cars. Both are real. But night is 4.7% of
crashes and trucks 2.0%, so neither moves the citywide expectation:
0.047 × 18.79% + 0.953 × 12.7% ≈ 13.0%, which is what we observe.

The model did learn the night effect — predicted 0.184 against 0.153 observed for
22:00–04:00, versus 0.137 / 0.119 at midday. It over-predicts across every band
because the test base rate (11.85%) sits below the training rate (13.36%), so the
ranking is informative while the level needs recalibrating on recent years.

A time-of-day multiplier could be applied — night is roughly 1.45× the citywide KSI
rate — but it would scale every segment identically and therefore leave the route
ordering, and the route, unchanged.

---
## 6. Limitations

1. **Conditional on a crash occurring.** This model estimates severity, not
   collision probability. It cannot answer "will I crash here" and is not intended
   to.
2. **Light and surface conditions are missing.** `data_pipeline.py` does not retain
   `ULICHTVERH` or `STRZUSTAND`, so the night effect is captured indirectly through
   `hour` rather than directly through observed darkness. Retaining those columns
   upstream would improve this model.
3. **Truck involvement cannot be a feature.** Nobody knows in advance which
   vehicle will be involved. The truck finding — 2% of crashes, 43% of cyclist
   deaths — is real but enters the product only as a conditional expectation, and
   the arithmetic shows it is diluted: 3.6% of morning crashes involve trucks at
   27.25% KSI, which moves the aggregate morning rate by roughly 0.6 points.
4. **Night severity is confounded.** Darkness, speed, alcohol and a different
   rider population all coincide, and alcohol and speed are absent from the data.
   The 18.79% night KSI rate cannot be attributed to darkness alone.
5. **E-bikes are not distinguished.** `IstRad` includes pedelecs, whose share grew
   substantially over 2018–2025, so the vehicle mix inside a single category
   shifted across the analysis window.
6. **Under-reporting is not uniform.** Single-bicycle falls reach police far less
   often than collisions, and that category has the second-highest severity
   (22.01% KSI). Its weight in this model is therefore understated.
7. **This is one factor of two.** Multiplied by a frequency model that still lacks
   an exposure denominator, the combined risk surface inherits that limitation.
8. **The KSI rate declines across the split.** Training years (2018–2023) average
   13.36% against 11.85% in the test years (2024–2025). The decline is gradual
   rather than a step change — 14.34% in 2018 falling to 11.73% in 2025, an 18%
   relative reduction (grouped 2018–2022 vs 2023–2025: 13.48% vs 12.13%, z = 3.75,
   p ≈ 0.0002) — which argues against a recording artefact but does not identify a
   cause. Infrastructure rollout under Berlin's 2018 Mobilitätsgesetz, growing
   e-bike share, COVID-era changes in volume and composition, and more complete
   recording of slight injuries are all consistent with it and cannot be separated
   here. The practical consequence is that the model over-predicts the level on
   recent data even where the ranking holds.